# Final Model

In this notebook, we evaluate the performance of our chosen model. We compare it's performance against the baseline. Afterwards, export the final fitted model.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, recall_score, 
                             f1_score, fbeta_score, confusion_matrix, log_loss, 
                             classification_report, average_precision_score, precision_recall_curve)

In [2]:
Xt_raw = pd.read_csv("../../data/cleaned_data/train.csv")
Xv_raw = pd.read_csv("../../data/cleaned_data/test.csv")

yt = Xt_raw["target"]
yv = Xv_raw["target"]

Xt_raw.drop(columns=["target"],inplace=True)
Xv_raw.drop(columns=["target"],inplace=True)

Xt_raw.drop(columns=["patient_nbr"],inplace=True)
Xv_raw.drop(columns=["patient_nbr"],inplace=True)

C:\Users\daoke\AppData\Local\Temp\ipykernel_24484\1168084511.py:1: DtypeWarning: Columns (16: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  Xt_raw = pd.read_csv("../../data/cleaned_data/train.csv")
C:\Users\daoke\AppData\Local\Temp\ipykernel_24484\1168084511.py:7: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  Xt_raw.drop(columns=["target"],inplace=True)
C:\Users\daoke\AppData\Local\Temp\ipykernel_24484\1168084511.py:8: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  Xv_raw.drop(columns=["target"],inplace=True)
C:\Users\daoke\AppData\Local\Temp\ipykernel_24484\1168084511.py:10: Pandas4Warning: The inplace keyw

In [3]:
BEST_TRIAL = 3410
V4_F2 = 0.439513
THRESHOLD = 0.4483490425281894
C = 1.5036376551308464
best_features = """
race age admission_type_id payer_code medical_specialty
metformin repaglinide nateglinide chlorpropamide glimepiride
glipizide glyburide tolbutamide pioglitazone rosiglitazone
acarbose miglitol metformin-pioglitazone diabetesmed alpha_glucosidase_inhibitor
biguanide sulfonylurea_gen2 thiazolidinedione number_outpatient_log1p number_emergency_log1p
number_inpatient_log1p eng_prior_inpatient_any eng_inpatient_emergency_interaction eng_labs_per_day eng_acute_visits
eng_any_prior_visit eng_acute_visit_share number_emergency_bin number_emergency_2plus number_inpatient_2plus
number_inpatient_5plus time_in_hospital_log1p time_in_hospital_sqrt time_in_hospital_squared num_lab_procedures_log1p
num_lab_procedures_sqrt num_lab_procedures_squared num_procedures_log1p num_procedures_sqrt num_medications_log1p
num_medications_sqrt number_outpatient_sqrt number_emergency_squared number_inpatient_sqrt number_diagnoses_log1p
number_diagnoses_sqrt number_diagnoses_squared time_in_hospital_x_num_lab_procedures time_in_hospital_x_num_procedures time_in_hospital_x_number_outpatient
time_in_hospital_x_number_emergency time_in_hospital_x_number_inpatient num_lab_procedures_x_num_procedures num_lab_procedures_x_number_emergency num_lab_procedures_x_number_inpatient
num_procedures_x_num_medications num_procedures_x_number_inpatient num_procedures_x_number_diagnoses num_medications_x_number_outpatient num_medications_x_number_inpatient
num_medications_x_number_diagnoses number_outpatient_x_number_inpatient number_emergency_x_number_inpatient number_emergency_x_number_diagnoses number_inpatient_x_number_diagnoses
num_lab_procedures_per_day eng_labs_per_medication eng_drugs_No eng_drugs_Steady eng_drugs_Up
eng_drugs_Down eng_drugs_changed metformin_changed repaglinide_active nateglinide_changed
chlorpropamide_active glimepiride_active acetohexamide_active acetohexamide_changed glipizide_changed
tolbutamide_active tolbutamide_changed pioglitazone_active pioglitazone_changed miglitol_changed
troglitazone_active examide_changed citoglipton_active insulin_changed glyburide-metformin_changed
glipizide-metformin_changed glimepiride-pioglitazone_active glimepiride-pioglitazone_changed metformin-pioglitazone_active metformin-pioglitazone_changed
meglitinide_count_Steady meglitinide_count_Up meglitinide_count_active sulfonylurea_gen1_count_Steady sulfonylurea_gen2_count_Steady
sulfonylurea_gen2_count_Up sulfonylurea_gen2_count_Down thiazolidinedione_count_Up alpha_glucosidase_count_active metformin_combinations_count_Up
metformin_combinations_count_Down metformin_combinations_count_active eng_distinct_diagnosis_groups eng_diagnosis_count_Circulatory eng_age_squared
eng_age_50plus eng_age_85plus diag_1_group_with_insulin diag_1_group_with_discharge_disposition_id age_with_discharge_disposition_id
insulin_with_change admission_type_id_with_admission_source_id diag_1_group_with_diag_2_group diabetesmed_with_a1cresult
""".split()

display(pd.DataFrame({"trial": [BEST_TRIAL], "validation_F2": [V4_F2],
                      "threshold": [THRESHOLD], "C": [C],
                      "class_weight": ["balanced"], "inputs": [len(best_features)]}))
display(pd.DataFrame({"selected_input": best_features}))

,trial,validation_F2,threshold,C,class_weight,inputs
0,3410,0.439513,0.448349,1.503638,balanced,124


,selected_input
0,race
1,age
2,admission_type_id
3,payer_code
4,medical_specialty
...,...
119,age_with_discharge_disposition_id
120,insulin_with_change
121,admission_type_id_with_admission_source_id
122,diag_1_group_with_diag_2_group


In [4]:
# carry over functions from other scripts

def group_diagnoses(df):
    ranges = {"Circulatory": (390, 459), "Respiratory": (460, 519),
        "Digestive": (520, 579), "Genitourinary": (580, 629),
        "Musculoskeletal": (710, 739), "Neoplasms": (140, 239), "Injury": (800, 999)}
    
    symptoms = {785: "Circulatory", 786: "Respiratory", 787: "Digestive", 788: "Genitourinary"}

    def categorize(value):
        if pd.isna(value):
            return "Missing"

        code = str(value).strip().upper()
        if code in {"", "?", "NAN", "NONE", "MISSING"}:
            return "Missing"

        # Supplementary V/E codes belong to the broad Other category.
        if code.startswith(("V", "E")):
            return "Other"

        try:
            number = float(code)
            if not 1 <= number < 1000:
                return "Other"
            code = int(number)  # 250.13 -> 250, 459.9 -> 459
        except ValueError:
            return "Other"

        if code == 250:
            return "Diabetes"
        if code in symptoms:
            return symptoms[code]
        for group, (lower, upper) in ranges.items():
            if lower <= code <= upper:
                return group
        return "Other"

    result = df.copy()
    for column in ("diag_1", "diag_2", "diag_3", "diaga_3"):
        if column in result.columns:
            result[column] = result[column].map(categorize)

    return result


def group_medications(df):
    groups = {
        "biguanide": ["metformin"],
        "meglitinide": ["repaglinide", "nateglinide"],
        "sulfonylurea_gen1": ["chlorpropamide", "acetohexamide", "tolbutamide", "tolazamide"],
        "sulfonylurea_gen2": ["glimepiride", "glipizide", "glyburide"],
        "thiazolidinedione": ["pioglitazone", "rosiglitazone", "troglitazone"],
        "alpha_glucosidase_inhibitor": ["acarbose", "miglitol"],
        "biguanide_sulfonylurea_gen2": ["glyburide-metformin", "glipizide-metformin"],
        "biguanide_thiazolidinedione": ["metformin-rosiglitazone", "metformin-pioglitazone"],
        "sulfonylurea_gen2_thiazolidinedione": ["glimepiride-pioglitazone"],
    }

    result = df.copy()
    priority = ["Up", "Down", "Steady"]

    for group, medications in groups.items():
        columns = df.columns.intersection(medications)
        if len(columns):
            values = df[columns]
            result = result.drop(columns=columns)
            result[group] = np.select([values.eq(status).any(axis=1) for status in priority], priority, default="No")

    return result



def engineer(df):
    visits = [
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
    ]

    # Allow diminishing effects from additional previous visits.
    for column in visits:
        df[f"{column}_log1p"] = np.log1p(df[column])

    df["eng_prior_visits_log1p"] = np.log1p(df[visits].sum(axis=1))
    df["eng_prior_inpatient_any"] = (df["number_inpatient"] > 0).astype(int)
    df["eng_prior_inpatient_2plus"] = (df["number_inpatient"] >= 2).astype(int)

    # Let combined inpatient/emergency utilization have an additional effect.
    df["eng_inpatient_emergency_interaction"] = (
        df["number_inpatient_log1p"]
        * df["number_emergency_log1p"]
    )

    # Distinguish intensity from total volume during the admission.
    days = df["time_in_hospital"].clip(lower=1)
    df["eng_medications_per_day"] = df["num_medications"] / days
    df["eng_labs_per_day"] = df["num_lab_procedures"] / days

    return df

In [5]:
visits = ["number_outpatient", "number_emergency", "number_inpatient"]
base_numeric = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
                *visits, "number_diagnoses"]
classes = {
    "meglitinide": ["repaglinide", "nateglinide"],
    "sulfonylurea_gen1": ["chlorpropamide", "acetohexamide", "tolbutamide", "tolazamide"],
    "sulfonylurea_gen2": ["glimepiride", "glipizide", "glyburide"],
    "thiazolidinedione": ["pioglitazone", "rosiglitazone", "troglitazone"],
    "alpha_glucosidase": ["acarbose", "miglitol"],
    "metformin_combinations": ["glyburide-metformin", "glipizide-metformin",
                               "metformin-rosiglitazone", "metformin-pioglitazone"],
}
frames = []
for raw in (Xt_raw, Xv_raw):
    X = raw.drop(columns=["encounter_id", "readmitted", "target", "patient_nbr"], errors="ignore").copy()
    grouped_diag, grouped_med = group_diagnoses(X), group_medications(X)
    for col in ("diag_1", "diag_2", "diag_3"):
        X[col + "_group"] = grouped_diag[col]
    for col in ("alpha_glucosidase_inhibitor", "biguanide", "sulfonylurea_gen2", "thiazolidinedione"):
        X[col] = grouped_med[col]
    X = engineer(X)

    total = X[visits].sum(axis=1, min_count=3)
    acute = X["number_inpatient"] + X["number_emergency"]
    extra = {"eng_acute_visits": acute, "eng_any_prior_visit": total.gt(0).astype(int),
             "eng_acute_visit_share": acute / (1 + total),
             "number_emergency_bin": pd.cut(X["number_emergency"], [-1, 0, 1, 2, 4, np.inf],
                                            labels=["0", "1", "2", "3-4", "5+"]).astype(object),
             "number_emergency_2plus": X["number_emergency"].ge(2).astype(int)}
    for n in (2, 5):
        extra[f"number_inpatient_{n}plus"] = X["number_inpatient"].ge(n).astype(int)

    for col in base_numeric:
        z = X[col].clip(lower=0)
        extra[col + "_log1p"] = np.log1p(z)
        extra[col + "_sqrt"] = np.sqrt(z)
        extra[col + "_squared"] = z ** 2
    for i, a in enumerate(base_numeric):
        for b in base_numeric[i + 1:]:
            name = a + "_x_" + b
            if name in best_features:
                extra[name] = extra[a + "_log1p"] * extra[b + "_log1p"]
    extra["num_lab_procedures_per_day"] = X["num_lab_procedures"] / X["time_in_hospital"].clip(lower=1)
    extra["eng_labs_per_medication"] = X["num_lab_procedures"] / (1 + X["num_medications"])

    drugs = raw.loc[:, "metformin":"metformin-pioglitazone"].columns.tolist()
    for status in ("No", "Steady", "Up", "Down"):
        extra["eng_drugs_" + status] = X[drugs].eq(status).sum(axis=1)
    extra["eng_drugs_changed"] = X[drugs].isin(["Up", "Down"]).sum(axis=1)
    for name in best_features:
        if name.endswith(("_active", "_changed")) and name.rsplit("_", 1)[0] in drugs:
            drug, mode = name.rsplit("_", 1)
            extra[name] = X[drug].isin(["Steady", "Up", "Down"] if mode == "active" else ["Up", "Down"]).astype(int)
    for name, members in classes.items():
        for status in ("Steady", "Up", "Down"):
            key = name + "_count_" + status
            if key in best_features:
                extra[key] = X[members].eq(status).sum(axis=1)
        key = name + "_count_active"
        if key in best_features:
            extra[key] = X[members].isin(["Steady", "Up", "Down"]).sum(axis=1)

    diag_groups = X[["diag_1_group", "diag_2_group", "diag_3_group"]]
    extra["eng_distinct_diagnosis_groups"] = diag_groups.nunique(axis=1)
    extra["eng_diagnosis_count_Circulatory"] = diag_groups.eq("Circulatory").sum(axis=1)
    age_mid = pd.to_numeric(X["age"].astype("string").str.extract(r"\[(\d+)-", expand=False), errors="coerce") + 5
    extra["eng_age_squared"] = age_mid ** 2
    for cutoff in (50, 85):
        extra[f"eng_age_{cutoff}plus"] = age_mid.ge(cutoff).astype(int)
    for a, b in (("diag_1_group", "insulin"), ("diag_1_group", "discharge_disposition_id"),
                 ("age", "discharge_disposition_id"), ("insulin", "change"),
                 ("admission_type_id", "admission_source_id"), ("diag_1_group", "diag_2_group"),
                 ("diabetesmed", "a1cresult")):
        extra[a + "_with_" + b] = X[a].fillna("Missing").astype(str) + "|" + X[b].fillna("Missing").astype(str)
    X = pd.concat([X, pd.DataFrame({k: v for k, v in extra.items() if k not in X.columns}, index=X.index)], axis=1)
    for col in ("diag_1", "diag_2", "diag_3", "admission_type_id",
                "discharge_disposition_id", "admission_source_id"):
        X[col] = X[col].astype(str).where(X[col].notna(), np.nan)
    for col in X.select_dtypes(exclude="number"):
        X[col] = X[col].astype(str).where(X[col].notna(), np.nan)
    frames.append(X)
Xt, Xv = frames

In [6]:
numeric = Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True,
                                             add_indicator=True)),
                    ("scale", StandardScaler())])
categorical = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Missing",
                                                 keep_empty_features=True)),
                        ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                 min_frequency=20))])

def make_pipeline(columns):
    prep = ColumnTransformer([(f"feature_{i}", numeric if pd.api.types.is_numeric_dtype(Xt[col])
                               else categorical, [col]) for i, col in enumerate(columns)],
                             sparse_threshold=1.0)
    model = LogisticRegression(solver="liblinear", C=C, class_weight="balanced",
                               max_iter=300, tol=1e-3)
    return Pipeline([("preprocess", prep), ("model", model)])

In [7]:
Xt_best, Xv_best = Xt[best_features], Xv[best_features]
print(f"{Xt_best.shape[1]} selected inputs, {len(Xt_best):,} training and {len(Xv_best):,} validation encounters")

124 selected inputs, 81,412 training and 20,354 validation encounters


In [8]:
full = make_pipeline(best_features)
full.fit(Xt_best, yt)
full_prob = full.predict_proba(Xv_best)[:, 1]

In [9]:
proba=full_prob 
threshold=THRESHOLD
y_pred = (proba >= threshold).astype(int)

print(f"Accuracy: {accuracy_score(yv, y_pred):.3f}")
print("Confusion matrix:\n", confusion_matrix(yv, y_pred))
print(classification_report(yv, y_pred))
print(f"F1 @{threshold}:", round(f1_score(yv, y_pred), 5))
f2 = fbeta_score(yv, y_pred, beta=2)
print(f"F2 @{threshold}:", round(f2, 5))

y_pred = (proba >= threshold).astype(int)
print("ROC-AUC:", round(roc_auc_score(yv, proba), 5))
print("PR-AUC: ", round(average_precision_score(yv, proba), 5))
p, r, thr = precision_recall_curve(yv, proba)
f1s = 2 * p * r / (p + r + 1e-12)
best = f1s[:-1].argmax()
print(f"\nBest-threshold F1: {f1s[best]:.5f} at threshold {thr[best]:.5f}.")

f2s = 5 * p * r / (4 * p + r + 1e-12)
best2 = f2s[:-1].argmax()
print(f"\nBest-threshold F2: {f2s[best2]:.5f} at threshold {thr[best2]:.5f}.")


Accuracy: 0.554
Confusion matrix:
 [[9697 8385]
 [ 686 1586]]
              precision    recall  f1-score   support

           0       0.93      0.54      0.68     18082
           1       0.16      0.70      0.26      2272

    accuracy                           0.55     20354
   macro avg       0.55      0.62      0.47     20354
weighted avg       0.85      0.55      0.63     20354

F1 @0.4483490425281894: 0.25909
F2 @0.4483490425281894: 0.41608
ROC-AUC: 0.67458
PR-AUC:  0.21737

Best-threshold F1: 0.28552 at threshold 0.51896.

Best-threshold F2: 0.42135 at threshold 0.40483.


# Result

In [10]:
print(f"The model obtained an F2 score of {round(f2, 5)} at threshold {threshold}.")

The model obtained an F2 score of 0.41608 at threshold 0.4483490425281894.
